# WildTrace Pipeline Driver

This notebook is the current driver for the ETL pipeline. It runs the existing stage scripts step by step so we can inspect outputs, calibrate configs, and understand each boundary before moving orchestration into Airflow, n8n, or another scheduler.

In [2]:
from pathlib import Path
import json
import subprocess
import yaml
from PIL import Image
from IPython.display import SVG, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'scripts').exists():
    for parent in REPO_ROOT.parents:
        if (parent / 'scripts').exists() and (parent / 'configs').exists():
            REPO_ROOT = parent
            break

PYTHON = REPO_ROOT / '.venv' / 'bin' / 'python'
SCRIPTS_DIR = REPO_ROOT / 'scripts'
CONFIG_DIR = REPO_ROOT / 'configs'

print('repo_root =', REPO_ROOT)
print('python =', PYTHON)
print('scripts_dir =', SCRIPTS_DIR)
print('config_dir =', CONFIG_DIR)

repo_root = /home/shra012/Workspace/WildTrace
python = /home/shra012/Workspace/WildTrace/.venv/bin/python
scripts_dir = /home/shra012/Workspace/WildTrace/scripts
config_dir = /home/shra012/Workspace/WildTrace/configs


In [3]:
def run_stage(script_name: str):
    script = SCRIPTS_DIR / script_name
    cmd = [str(PYTHON), str(script), '--repo-root', str(REPO_ROOT), '--config-dir', str(CONFIG_DIR)]
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, check=True)


def load_ndjson(path: Path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

## Inspect Configs

Tune these first before running a larger fetch.

In [4]:
for name in ['datasets.yaml', 'storage.yaml', 'quality.yaml', 'models.yaml', 'export.yaml']:
    path = CONFIG_DIR / name
    print(f'\n## {name}')
    print(yaml.safe_dump(yaml.safe_load(path.read_text()), sort_keys=False))


## datasets.yaml
task_type: drawing
source_dataset: openimages
categories:
- name: Cat
  limit: 300
- name: Dog
  limit: 300
- name: Horse
  limit: 300
- name: Bird
  limit: 300
- name: Butterfly
  limit: 300
- name: Fish
  limit: 300
fetch:
  source_mode: official
  records_path: null
  records: []
  timeout_seconds: 30
  user_agent: WildTraceETL/0.1
  manifest_path: raw_data/bronze/manifests/openimages_fetch.ndjson
  latest_view_path: raw_data/bronze/manifests/openimages_fetch_latest.ndjson
  official:
    splits:
    - train
    - validation
    require_masks: true
    class_descriptions_url: https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv
    image_base_url_template: https://open-images-dataset.s3.amazonaws.com/{split}/{image_id}.jpg
    image_info_urls:
      train: https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv
      validation: https://storage.googleapis.com/openimages/2018_04/validation/validatio

## Step 1: Fetch Open Images

In [5]:
run_stage('fetch_openimages.py')

fetch_ledger = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'openimages_fetch.ndjson'
fetch_latest = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'openimages_fetch_latest.ndjson'
fetch_rows = load_ndjson(fetch_ledger)
fetch_latest_rows = load_ndjson(fetch_latest)
print('fetch ledger rows =', len(fetch_rows))
print('fetch latest rows =', len(fetch_latest_rows))
fetch_latest_rows[:3]

Running: /home/shra012/Workspace/WildTrace/.venv/bin/python /home/shra012/Workspace/WildTrace/scripts/fetch_openimages.py --repo-root /home/shra012/Workspace/WildTrace --config-dir /home/shra012/Workspace/WildTrace/configs
fetch ledger rows = 1500
fetch latest rows = 1500


[{'asset_version': 1,
  'category': 'Bird',
  'failure_reason': None,
  'fetch_status': 'success',
  'fetched_at': '2026-03-28T01:12:41.518579+00:00',
  'image_checksum': 'f12acf4aba7826386e9a8637df0296d7ca96b8a08da89baab8ec463007924490',
  'image_path': 'raw_data/bronze/openimages/images/Bird/0063b9ec94872008/v0001.jpg',
  'image_uri': 'local://raw_data/bronze/openimages/images/Bird/0063b9ec94872008/v0001.jpg',
  'license': 'https://creativecommons.org/licenses/by/2.0/',
  'mask_archive_url': 'https://storage.googleapis.com/openimages/v5/train-masks/train-masks-c.zip',
  'mask_checksum': '383767203e053bc19751e2e20c4edb732dcd58db66e98b5599f000dee0d298eb',
  'mask_path': 'raw_data/bronze/openimages/masks/Bird/0063b9ec94872008/v0001.png',
  'mask_path_in_archive': 'c3e3660f0f592431_m015p6_3bcb0e96.png',
  'mask_source_url': None,
  'mask_uri': 'local://raw_data/bronze/openimages/masks/Bird/0063b9ec94872008/v0001.png',
  'metadata_path': 'raw_data/bronze/openimages/metadata/Bird/0063b9ec9

## Step 2: Ingest Bronze Assets

In [ ]:
run_stage('ingest_openimages.py')

ingest_ledger = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'openimages_ingest.ndjson'
ingest_latest = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'openimages_ingest_latest.ndjson'
ingest_rows = load_ndjson(ingest_ledger)
ingest_latest_rows = load_ndjson(ingest_latest)
print('ingest ledger rows =', len(ingest_rows))
print('ingest latest rows =', len(ingest_latest_rows))
ingest_latest_rows[:3]

Running: /home/shra012/Workspace/WildTrace/.venv/bin/python /home/shra012/Workspace/WildTrace/scripts/ingest_openimages.py --repo-root /home/shra012/Workspace/WildTrace --config-dir /home/shra012/Workspace/WildTrace/configs
ingest ledger rows = 1500
ingest latest rows = 1500


[{'asset_version': 1,
  'category': 'Bird',
  'checksum': 'f12acf4aba7826386e9a8637df0296d7ca96b8a08da89baab8ec463007924490',
  'failure_reason': None,
  'height': 1024,
  'image_path': 'raw_data/bronze/openimages/images/Bird/0063b9ec94872008/v0001.jpg',
  'image_uri': 'local://raw_data/bronze/openimages/images/Bird/0063b9ec94872008/v0001.jpg',
  'ingest_status': 'success',
  'ingest_timestamp': '2026-03-28T02:07:24.855302+00:00',
  'license': 'https://creativecommons.org/licenses/by/2.0/',
  'lineage': {'fetch_record_id': '753b14d152b9e6ea',
   'ingest_config_hash': 'b04b3c85419d1e0c'},
  'mask_path': 'raw_data/bronze/openimages/masks/Bird/0063b9ec94872008/v0001.png',
  'mask_uri': 'local://raw_data/bronze/openimages/masks/Bird/0063b9ec94872008/v0001.png',
  'metadata_path': 'raw_data/bronze/openimages/metadata/Bird/0063b9ec94872008/v0001.json',
  'record_id': '753b14d152b9e6ea',
  'sample_id': '0063b9ec94872008',
  'source_dataset': 'openimages',
  'source_image_id': 'c3e3660f0f59243

: 

## Step 3: Validate Bronze

In [ ]:
run_stage('validate_bronze.py')

validation_path = REPO_ROOT / 'raw_data' / 'bronze' / 'manifests' / 'bronze_validation.ndjson'
validation_rows = load_ndjson(validation_path)
validation_rows[:3]

## Step 4: Normalize To Silver

In [ ]:
run_stage('normalize_to_silver.py')

silver_path = REPO_ROOT / 'processed' / 'silver' / 'qa' / 'silver_samples.ndjson'
silver_rows = load_ndjson(silver_path)
silver_rows[:2]

In [ ]:
if silver_rows:
    sample = silver_rows[0]
    display(Image.open(REPO_ROOT / sample['image_path']))
    if sample['isolated_path']:
        display(Image.open(REPO_ROOT / sample['isolated_path']))

## Step 5: Outline Inference

In [ ]:
run_stage('run_outline_inference.py')

outline_path = REPO_ROOT / 'processed' / 'gold' / 'outlines' / 'outline_inference.ndjson'
outline_rows = load_ndjson(outline_path)
outline_rows[:2]

## Step 6: Refine Outlines

In [ ]:
run_stage('refine_outlines.py')

refined_path = REPO_ROOT / 'processed' / 'gold' / 'trajectories' / 'refined_outlines.ndjson'
refined_rows = load_ndjson(refined_path)
refined_rows[:2]

## Step 7: Export Gold NDJSON

In [ ]:
run_stage('export_gold_ndjson.py')

gold_path = REPO_ROOT / 'processed' / 'gold' / 'ndjson' / 'gold_samples.ndjson'
gold_rows = load_ndjson(gold_path)
gold_rows[:2]

In [ ]:
if gold_rows:
    first = gold_rows[0]
    display(SVG(filename=str(REPO_ROOT / first['gold_refs']['svg_path'])))
    first['trajectory']

## Step 8: Generate Dataset Report

In [ ]:
run_stage('generate_dataset_report.py')

report_json = REPO_ROOT / 'artifacts' / 'reports' / 'dataset_report.json'
report_md = REPO_ROOT / 'artifacts' / 'reports' / 'dataset_report.md'
print(report_json.read_text(encoding='utf-8'))
print('\n---\n')
print(report_md.read_text(encoding='utf-8'))